# Art Attack SDXL LoRA Training

This notebook trains a LoRA for the Art Attack app using the official Hugging Face Diffusers `train_text_to_image_lora_sdxl.py` script.

Before running training, create the quick dataset by running `python lora/prepare_dataset.py` from the project root.

```text
train_dataset_300/
  images/
    0001.png
    0002.png
  metadata.jsonl
```

Each metadata line should contain `file_name` and `text`, and every caption should start with `artattackstyle,`.

In [ ]:
# This cell defines all paths and training settings in one place.
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
DATASET_DIR = NOTEBOOK_DIR / "train_dataset_300"
IMAGES_DIR = DATASET_DIR / "images"
METADATA_FILE = DATASET_DIR / "metadata.jsonl"
OUTPUT_DIR = NOTEBOOK_DIR / "trained_lora"
SCRIPT_DIR = NOTEBOOK_DIR / "diffusers_training_script"
TRAIN_SCRIPT = SCRIPT_DIR / "train_text_to_image_lora_sdxl.py"

BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
VAE_MODEL = "madebyollin/sdxl-vae-fp16-fix"
TRIGGER_WORD = "artattackstyle"

RESOLUTION = 512
TRAIN_STEPS = 300
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = "1e-4"
LORA_RANK = 8

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", DATASET_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# This cell installs extra packages needed by the official Diffusers LoRA trainer.
# Run it once. Restart the notebook kernel if Jupyter asks you to.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "..\\requirements.txt"], check=True)

In [ ]:
# This cell downloads the official SDXL LoRA training script from Hugging Face Diffusers.
from urllib.request import urlretrieve

SCRIPT_URL = "https://raw.githubusercontent.com/huggingface/diffusers/v0.38.0/examples/text_to_image/train_text_to_image_lora_sdxl.py"

if not TRAIN_SCRIPT.exists():
    urlretrieve(SCRIPT_URL, TRAIN_SCRIPT)
    print("Downloaded:", TRAIN_SCRIPT)
else:
    print("Training script already exists:", TRAIN_SCRIPT)

## Dataset Format

The official trainer can load a local `imagefolder` dataset. This notebook expects:

```json
{"file_name":"images/0001.png","text":"artattackstyle, full body dark fantasy knight, black armor"}
```

If you manually create `.txt` caption files next to images instead, run the optional helper cell below to create `metadata.jsonl` automatically.

In [ ]:
# Optional helper: create metadata.jsonl from images and matching .txt caption files.
# Example: images/0001.png should have images/0001.txt beside it.
import json

image_extensions = {".png", ".jpg", ".jpeg", ".webp"}
rows = []

for image_path in sorted(IMAGES_DIR.glob("*")):
    if image_path.suffix.lower() not in image_extensions:
        continue

    caption_path = image_path.with_suffix(".txt")
    if caption_path.exists():
        caption = caption_path.read_text(encoding="utf-8").strip()
    else:
        caption = "full body dark fantasy character, realistic character concept art"

    if not caption.lower().startswith(TRIGGER_WORD):
        caption = f"{TRIGGER_WORD}, {caption}"

    rows.append({"file_name": f"images/{image_path.name}", "text": caption})

if rows:
    with METADATA_FILE.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(rows)} rows to {METADATA_FILE}")
else:
    print("No images found. Run python lora/prepare_dataset.py from the project root first.")

In [ ]:
# This cell validates the dataset before training starts.
import json

assert DATASET_DIR.exists(), f"Missing dataset folder: {DATASET_DIR}"
assert IMAGES_DIR.exists(), f"Missing images folder: {IMAGES_DIR}"
assert METADATA_FILE.exists(), f"Missing metadata file: {METADATA_FILE}"

missing_files = []
missing_trigger = []
row_count = 0

with METADATA_FILE.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue
        row_count += 1
        row = json.loads(line)
        image_file = DATASET_DIR / row["file_name"]
        caption = row.get("text", "")
        if not image_file.exists():
            missing_files.append((line_number, str(image_file)))
        if not caption.lower().startswith(TRIGGER_WORD):
            missing_trigger.append((line_number, caption[:80]))

print("Rows:", row_count)
print("Missing files:", len(missing_files))
print("Captions missing trigger word:", len(missing_trigger))

if missing_files[:5]:
    print("First missing files:", missing_files[:5])
if missing_trigger[:5]:
    print("First missing triggers:", missing_trigger[:5])

assert row_count > 0, "metadata.jsonl has no rows."
assert not missing_files, "Some metadata rows point to missing images."
assert not missing_trigger, f"Some captions do not start with {TRIGGER_WORD}."

In [ ]:
# This cell checks that Accelerate is installed before training.
# The training cell passes mixed precision directly, so no interactive config is needed.
import accelerate

print("accelerate", accelerate.__version__)

In [ ]:
# This cell trains the LoRA using the official Diffusers SDXL LoRA trainer.
# On an RTX 4070 Ti SUPER, start with 1500 steps and adjust after testing.
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "accelerate.commands.launch",
    "--mixed_precision=fp16",
    str(TRAIN_SCRIPT),
    "--pretrained_model_name_or_path", BASE_MODEL,
    "--pretrained_vae_model_name_or_path", VAE_MODEL,
    "--train_data_dir", str(DATASET_DIR),
    "--image_column", "image",
    "--caption_column", "text",
    "--resolution", str(RESOLUTION),
    "--center_crop",
    "--random_flip",
    "--train_batch_size", str(BATCH_SIZE),
    "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--max_train_steps", str(TRAIN_STEPS),
    "--learning_rate", LEARNING_RATE,
    "--rank", str(LORA_RANK),
    "--lr_scheduler", "cosine",
    "--lr_warmup_steps", "100",
    "--checkpointing_steps", "500",
    "--num_validation_images", "0",
    "--output_dir", str(OUTPUT_DIR),
]

subprocess.run(command, check=True)

In [ ]:
# This cell checks that training produced LoRA weights.
from pathlib import Path

weights = list(OUTPUT_DIR.glob("*.safetensors"))
print("LoRA output folder:", OUTPUT_DIR)
print("Weights found:", [w.name for w in weights])

if not weights:
    print("No .safetensors file found yet. Check the training log above for errors.")

## Use In Art Attack

After training finishes, launch the app and use:

```text
LoRA path: lora/trained_lora
LoRA trigger: artattackstyle
```

First test with `Sketch conditioning = none`, then test with `pose`.